In [ ]:

# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────


# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# 2. TEMPORAL FEATURES
# ─────────────────────────────────────────────
print("Building temporal features …")

# Holiday proximity

# ─────────────────────────────────────────────
# 3. LAG FEATURES
# ─────────────────────────────────────────────
print("Building lag features …")

# ─────────────────────────────────────────────
# 4. ROLLING FEATURES (shift=7 to avoid leakage)
# ─────────────────────────────────────────────
print("Building rolling features …")


# ─────────────────────────────────────────────
# 5. EWM FEATURES
# ─────────────────────────────────────────────
print("Building EWM features …")


# ─────────────────────────────────────────────
# 6. PRICE FEATURES
# ─────────────────────────────────────────────
print("Building price features …")

# ─────────────────────────────────────────────
# 7. SPATIAL & COMPETITION FEATURES
# ─────────────────────────────────────────────
print("Building spatial features …")

# ─────────────────────────────────────────────
# 8. LOYALTY DAY INTERACTIONS
# ─────────────────────────────────────────────
print("Building loyalty features …")

# ─────────────────────────────────────────────
# 9. NEW PRODUCT HANDLING
# ─────────────────────────────────────────────
print("Handling new products …")

# ─────────────────────────────────────────────
# 10. ENCODE CATEGORICALS
# ─────────────────────────────────────────────
print("Encoding categoricals …")


# ─────────────────────────────────────────────
# 11. SPLIT BACK & DEFINE FEATURES
# ─────────────────────────────────────────────
# ─────────────────────────────────────────────
# 12. TARGET ENCODING HELPER
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# 13. CROSS-VALIDATION
# ─────────────────────────────────────────────


Loading data …
Building temporal features …
Building lag features …
Building rolling features …
Building EWM features …
Building price features …
Building spatial features …
Building loyalty features …
Handling new products …
Encoding categoricals …

→ Base features : 85
→ TE features   : 4
→ Total features: 89

── Training CatBoost ──
  Fold 1 — MAE: 0.9217 | best iter: 1941
  Fold 2 — MAE: 0.9517 | best iter: 2075
  Fold 3 — MAE: 0.9393 | best iter: 2299
  Fold 4 — MAE: 0.9347 | best iter: 2056
  Fold 5 — MAE: 0.9310 | best iter: 2916

  CatBoost OOF MAE: 0.9357

── Training XGBoost ──


TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:

# ── XGBoost ───────────────────────────────────
# ─────────────────────────────────────────────
# 14. WEIGHTED ENSEMBLE (CV predictions)
# ─────────────────────────────────────────────
print("\n── Building ensemble ──")

# ─────────────────────────────────────────────
# 15. FINAL MODELS (trained on full train data)
# ─────────────────────────────────────────────
print("\n── Fitting final models on full training data …")

# Compute TE on full train set for final models
tr_final = tr.copy()
te_final = te.copy()   # already has averaged TE from CV above

for key_cols in TE_CONFIGS:
    col = f"te_{'_'.join(key_cols)}"
    tr_final[col] = target_encode(tr_final, tr_final, key_cols)

final_cb = CatBoostRegressor(**{**cb_params,
                                 "iterations": 2000,
                                 "early_stopping_rounds": None})
final_cb.fit(tr_final[ALL_FEATURES], y_train)

final_xgb = xgb.XGBRegressor(**{**xgb_params, "n_estimators": 2000})
final_xgb.fit(tr_final[ALL_FEATURES], y_train)

# ─────────────────────────────────────────────
# 16. RECURSIVE 14-DAY PREDICTION
#     For each test day (in order), patch lag features
#     using previously predicted values, then predict.
#     Blended 50/50 with CV ensemble predictions.
# ─────────────────────────────────────────────
print("\n── Recursive 14-day prediction ──")

# Seed lookup with recent training actuals
last_train_rows = (
    train.sort_values("date")
         .groupby(["store_id", "product_id"])
         .tail(HORIZON * 3)
         [["store_id", "product_id", "date", "demand"]]
)
demand_lookup = {}
for row in last_train_rows.itertuples(index=False):
    demand_lookup[(row.store_id, row.product_id, row.date)] = row.demand

test_dates     = sorted(te_final["date"].unique())
recursive_preds = {}



── Training XGBoost ──
[0]	validation_0-mae:1.32937
[500]	validation_0-mae:0.92271


KeyboardInterrupt: 

In [11]:

for day_offset, pred_date in enumerate(test_dates):
    day_df = te_final[te_final["date"] == pred_date].copy()

    # Patch lag columns that now have real (predicted) values available
    for lag in [1, 2, 3, 4, 5, 6, 7, 14]:
        lag_col  = f"lag_{lag}"
        lag_date = pred_date - pd.Timedelta(days=lag)
        if lag_col not in day_df.columns:
            continue
        day_df[lag_col] = day_df.apply(
            lambda r, ld=lag_date, lc=lag_col: demand_lookup.get(
                (r["store_id"], r["product_id"], ld), r[lc]
            ), axis=1
        )

    cb_p  = final_cb.predict(day_df[ALL_FEATURES]).clip(min=0)
    xgb_p = final_xgb.predict(day_df[ALL_FEATURES]).clip(min=0)
    day_pred = (w_cb * cb_p + w_xgb * xgb_p) / w_sum

    for i, row_id in enumerate(day_df["row_id"].values):
        sp  = day_df["store_id"].iloc[i]
        pp  = day_df["product_id"].iloc[i]
        demand_lookup[(sp, pp, pred_date)] = day_pred[i]
        recursive_preds[row_id] = day_pred[i]

    print(f"  Day {day_offset+1:2d}/{HORIZON}  ({pred_date.date()})  "
          f"mean_pred={day_pred.mean():.2f}")

# ─────────────────────────────────────────────
# 17. BLEND CV ENSEMBLE + RECURSIVE
# ─────────────────────────────────────────────
te_final["recursive_pred"] = te_final["row_id"].map(recursive_preds).clip(lower=0)

# 50/50 blend: CV ensemble brings diversity, recursive brings lag accuracy


final_preds = (0.5 * base_preds + 0.5 * te_final["recursive_pred"].values).clip(min=0)

# ─────────────────────────────────────────────
# 18. GENERATE SUBMISSION
# ─────────────────────────────────────────────
print(f"\n── Writing submission to {OUT_PATH} ──")

te_result           = te_final[["row_id"]].copy()
te_result["demand"] = final_preds

sub_out             = sub[["row_id"]].merge(te_result, on="row_id", how="left")
sub_out["demand"]   = sub_out["demand"].fillna(0).clip(lower=0)

sub_out.to_csv(OUT_PATH, index=False)
print(f"  Done! Shape: {sub_out.shape}")
print(sub_out.head(10))

# ─────────────────────────────────────────────
# 19. FEATURE IMPORTANCE (CatBoost)
# ─────────────────────────────────────────────
print("\n── Top 25 features (CatBoost) ──")
importance = pd.Series(
    final_cb.get_feature_importance(), index=ALL_FEATURES
).sort_values(ascending=False)
print(importance.head(25).to_string())
print("\n✅ Pipeline v2 complete.")

  Day  1/14  (2025-05-11)  mean_pred=1.40
  Day  2/14  (2025-05-12)  mean_pred=1.31
  Day  3/14  (2025-05-13)  mean_pred=1.33
  Day  4/14  (2025-05-14)  mean_pred=1.38
  Day  5/14  (2025-05-15)  mean_pred=1.58
  Day  6/14  (2025-05-16)  mean_pred=1.82
  Day  7/14  (2025-05-17)  mean_pred=1.74
  Day  8/14  (2025-05-18)  mean_pred=1.30
  Day  9/14  (2025-05-19)  mean_pred=1.20
  Day 10/14  (2025-05-20)  mean_pred=1.15
  Day 11/14  (2025-05-21)  mean_pred=1.19
  Day 12/14  (2025-05-22)  mean_pred=1.40
  Day 13/14  (2025-05-23)  mean_pred=1.60
  Day 14/14  (2025-05-24)  mean_pred=1.60

── Writing submission to submission.csv ──
  Done! Shape: (8120, 2)
   row_id    demand
0       0  5.819884
1       1  3.239814
2       2  1.148705
3       3  0.125745
4       4  0.237534
5       5  0.092216
6       6  0.132093
7       7  2.411505
8       8  1.672384
9       9  0.047921

── Top 25 features (CatBoost) ──
ewm_mean_7                15.138760
lag_1                     11.962303
ewm_mean_14      